In [237]:
%pip install python-dotenv openai numpy pydantic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [238]:
import numpy as np
from dotenv import load_dotenv
from pydantic import BaseModel

from openai import OpenAI

import os
from enum import Enum

In [239]:
load_dotenv()

True

In [240]:
LLM_API_URL = os.getenv("LLM_API_URL")
LLM_API_TOKEN = os.getenv("LLM_API_TOKEN")

print(f"LLM_API_URL: {LLM_API_URL}")
print(f"LLM_API_TOKEN: {LLM_API_TOKEN}")

MODEL = "google/gemma-4-e4b"

LLM_API_URL: http://172.26.128.1:1234/v1
LLM_API_TOKEN: sk-lm-VswvHhzP:xhpy0T4I5gJxaCwNGT44


In [241]:
client = OpenAI(
    base_url=LLM_API_URL,
    api_key=LLM_API_TOKEN
)
'''
response = client.responses.create(
    model=MODEL,
    instructions="You are a coding assistant that talks like a pirate.",
    input="How do I check if a Python object is an instance of a class?",
)

print(response.output_text)'''

'\nresponse = client.responses.create(\n    model=MODEL,\n    instructions="You are a coding assistant that talks like a pirate.",\n    input="How do I check if a Python object is an instance of a class?",\n)\n\nprint(response.output_text)'

# Modélisation du monde

In [242]:
VOID = 0
PLAYER = 1
ENNEMY = 2
GOLD = 3

SYMBOLS = {VOID: "·", PLAYER: "👤", ENNEMY: "👹", GOLD: "💰"}

In [243]:
#initial_map = np.random.randint(0, 4, size=(7, 7))

initial_map = np.array([
    [0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 2, 0, 3], # (1, 1) # (1, 4) # (1, 6)
    [0, 0, 0, 3, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 3], # (5, 6)
    [0, 0, 0, 0, 0, 0, 0],
])

initial_map

array([[0, 0, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 2, 0, 3],
       [0, 0, 0, 3, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 3],
       [0, 0, 0, 0, 0, 0, 0]])

# Couche de contrat

In [244]:
H = "HAUT"
B = "BAS"
G = "GAUCHE"
D = "DROITE"

'''
    H = "TOP"
    B = "DOWN"
    G = "LEFT"
    D = "RIGHT"
'''

class Direction(str, Enum):
    HAUT = H
    BAS = B
    GAUCHE = G
    DROITE = D
    
class PlayerDecision(BaseModel):
    #directionJustification: str
    direction : Direction

MOVES = {
    H: (-1, 0),
    B: (1, 0),
    G: (0, -1),
    D: (0, 1),
}

# Moteur de perception

In [245]:
def localize(world_map, entity):
    positions = np.argwhere(world_map == entity)
    return positions

In [246]:
def compute_distances(entities_positions, reference_pos):
    if(len(entities_positions) == 0):
        return np.array([])
    
    v = entities_positions - reference_pos
    distances = np.linalg.norm(v, axis=1)
    
    return np.round(distances, 2)

In [247]:
def perception(world_map):
    player_position = localize(world_map, PLAYER)[0]
    gold_positions = localize(world_map, GOLD)
    ennemies_positions = localize(world_map, ENNEMY)

    gold_distances = compute_distances(gold_positions, player_position)
    ennemies_distances = compute_distances(ennemies_positions, player_position)

    # perception directionnelle -> delta signe vers l'or le plus proche
    nearest_gold_delta = {"row": 0, "col": 0}
    if len(gold_positions) > 0:
        nearest_idx = int(np.argmin(gold_distances))
        d_row, d_col = gold_positions[nearest_idx] - player_position
        nearest_gold_delta = {"row": int(d_row), "col": int(d_col)}

    return {
        "ennemies_distances": ennemies_distances.tolist(),
        "ennemies_count": len(ennemies_positions),
        "gold_distances": gold_distances.tolist(),
        "gold_count": len(gold_positions),
        "nearest_gold_delta": nearest_gold_delta,
    }

In [248]:
def show_map(world_map):
    for row in world_map:
        print("\t".join(SYMBOLS.get(cell, "?") for cell in row))
    print('---------------------------------------------------')
    
show_map(initial_map)

·	·	·	·	·	·	·
·	👤	·	·	👹	·	💰
·	·	·	💰	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
---------------------------------------------------


# Moteur de déplacement

In [249]:
def allowed_move(world_map: np.ndarray, pos):
    n_rows, n_cols = world_map.shape
    r, c = pos
    
    if r < 0 or r >= n_rows or c < 0 or c >= n_cols:
        return False
    
    return world_map[r, c] in (VOID, GOLD)

In [250]:
def move(world_map: np.ndarray, old_pos, new_pos):
    move_result  = {
        "gold_collected": False,
        "new_pos": old_pos
    }
    
    if not allowed_move(world_map, new_pos):
        return move_result
    
    entity = world_map[old_pos[0], old_pos[1]]
    target = world_map[new_pos[0], new_pos[1]]
    world_map[old_pos[0], old_pos[1]] = VOID
    world_map[new_pos[0], new_pos[1]] = entity
    
    move_result["new_pos"] = new_pos
    
    if target == GOLD:
        move_result["gold_collected"] = True
        
    return move_result

# Moteur de décision

In [251]:
def decide(player_perception) -> PlayerDecision | None:
    delta = player_perception["nearest_gold_delta"]
    prompt = f"""
    # Contexte
    - Tu es un joueur qui veut ramasser le plus d'or possible
    
    # System de coordonnées
    - `row` augmente ver le BAS, diminue vers le HAUT.
    - `col` augmente vers la DORITE, diminue ver la GAUCHE.
    - `nearest_gold_delta` = position(or) - position(joueur).
    
    # Regles de decision (deterministes, applique dans cet ordre)
    1. Si delta.row < 0 -> HAUT
    2. Sinon si delta.row > 0 -> BAS
    3. Sinon si delta.col < 0 -> GAUCHE
    4. Sinon si delta.col > 0 -> DROITE 
    
    # Objectif
    - Trouve le plus court chemin vers l'or
    
    # Perception
    - nearest_gold_delta: row={delta['row']}, col={delta['col']}
    - donnees brutes: {player_perception}
    """
    
    #print(prompt)
    print(str(player_perception))
    
    response = client.beta.chat.completions.parse(
        model = MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format = PlayerDecision,
        temperature = 0.3
    )
    
    return response.choices[0].message.parsed or None

# Game loop (simulation)

In [252]:
def game_loop(world_map: np.ndarray, max_turns = 10):
    world_map = world_map.copy()
    move_history = []
    
    for turn in range(max_turns):
        print(f"\n==================== [Turn {turn + 1}] ====================")
        show_map(world_map)
        
        player_pos = localize(world_map, PLAYER)[0]
        
        p = perception(world_map)
        #p["move_history"] = move_history
        
        decision : PlayerDecision | None = decide(p)
        
        if decision is not None:
            print(f"\t → LLM decision: {decision.direction.value}")
            #print(f"\t → LLM justification: {decision.directionJustification}")
            
            move_history.append(decision.direction.value)
            
            d_row, d_col = MOVES[decision.direction.value]
            new_pos = (player_pos[0] + d_row, player_pos[1] + d_col)
            
            move_result = move(world_map, player_pos, new_pos)
            if move_result["gold_collected"]:
                print("FOUND GOLD !!!")
                break
            
            new_pos = move_result["new_pos"]

In [253]:
game_loop(world_map=initial_map, max_turns=10)


==================== [Turn 1] ====================
·	·	·	·	·	·	·
·	👤	·	·	👹	·	💰
·	·	·	💰	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
---------------------------------------------------
{'ennemies_distances': [3.0], 'ennemies_count': 1, 'gold_distances': [5.0, 2.24, 6.4], 'gold_count': 3, 'nearest_gold_delta': {'row': 1, 'col': 2}}
	 → LLM decision: DROITE

==================== [Turn 2] ====================
·	·	·	·	·	·	·
·	·	👤	·	👹	·	💰
·	·	·	💰	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
---------------------------------------------------
{'ennemies_distances': [2.0], 'ennemies_count': 1, 'gold_distances': [4.0, 1.41, 5.66], 'gold_count': 3, 'nearest_gold_delta': {'row': 1, 'col': 1}}
	 → LLM decision: BAS

==================== [Turn 3] ====================
·	·	·	·	·	·	·
·	·	·	·	👹	·	💰
·	·	👤	💰	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
---------------------------------------------------
{'ennemies_distances': [2.24], 'ennemies_count': 1

# Todo 01/07